[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-09-subword-tokenization.ipynb)

# Lecture 9: what your model cannot say, and subwords that can

Your A8 model has a **fixed vocabulary**: every word it saw in training, and nothing else. A
word it never saw becomes `<unk>`, and it can never produce that word in a translation, however
well it trains. This activity measures how much of *your* test set that is, then builds a
subword vocabulary that has no such gap.

When it ends you will have, in `MyDrive/CS479/assignment8`:

1. **A SentencePiece model trained on your A8 training split**, at a vocabulary size you chose:
   `spm.model` and `spm.vocab`.
2. **Two numbers side by side**, from a counting function you wrote: how many test words your
   word vocabulary cannot represent, and how many your subword vocabulary cannot. The second
   should be zero.
3. **A round-trip check**: text encoded to pieces and decoded back, unchanged.
4. **The settings A9 needs**, printed for your data, and your prediction of what they will do to
   your BLEU.

**Nothing here retrains a model.** That is Assignment 9, which reuses your A8 notebook almost
unchanged. This runs on a CPU, so no GPU is needed today.

**Running in Google Colab?** Run the next two cells. The first installs TorchLingo; the second
connects Google Drive, where your A8 files are. There is nothing to uncomment.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — the in-class activity for Lecture 9 (*Handling Morphology and Terminology in NMT*). A head start on **A9**.

In [ ]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
import torchlingo

In [ ]:
# Connect Google Drive, where the A8 kickoff notebook wrote your split.
from torchlingo.colab import setup

env = setup(drive=True)
A8_DIR = env.drive_dir / "CS479" / "assignment8"

missing = [n for n in ("train.tsv", "val.tsv", "test.tsv") if not (A8_DIR / n).exists()]
if missing:
    raise SystemExit(
        f"Not found in {A8_DIR}: {', '.join(missing)}. These are written by Step 5 of the A8 "
        "kickoff notebook. Run that far first, or point A8_DIR at wherever your split is."
    )

import pandas as pd
from torchlingo.config import Config
from torchlingo.data_processing import NMTDataset
from torchlingo.data_processing.vocab import CharVocab, SentencePieceVocab
from torchlingo.diagnostics import uniform_loss
from torchlingo.preprocessing import train_sentencepiece

test = pd.read_csv(A8_DIR / "test.tsv", sep="\t", dtype=str, keep_default_na=False)
print(f"Found your A8 split in {A8_DIR}; {len(test):,} test pairs.")

# A language written without spaces between words averages one or two "words" a line. For it,
# A8's NO_SPACES switch made the target vocabulary characters, so that is what this compares.
NO_SPACES = test.tgt.str.split().str.len().mean() < 3
UNITS = "character" if NO_SPACES else "word"
if NO_SPACES:
    print("Your language is written without spaces: your A8 model's target vocabulary was "
          "characters (NO_SPACES = True), so characters are what the pieces are compared with.")

## Step 1: what your word vocabulary cannot say

This rebuilds the exact vocabulary your A8 model uses (every word in the training split) and
asks how many **test** words fall outside it. Each one is an `<unk>`: on the source side the model
cannot read it, and on the target side it cannot write it, ever.

**If your language is written without spaces**, your A8 model's target side was characters, not
words, and that is what this rebuilds: a character seen fewer than twice in training is your
`<unk>`. Read "word" as "character" for the target side, here and below.

The function that counts them is yours to write, and it is four lines. `vocab.encode(sentence,
add_special_tokens=False)` gives one sentence's token ids; `vocab.unk_idx` is the id every unknown
word becomes. Return how many ids are `unk_idx` and how many ids there are in all. The cell
checks your function on a case whose answer is known before trusting it on your data.

**Before you run it, write down a guess** at the target-side percentage. There is no right
guess; there is a number you can be surprised by.

In [ ]:
if NO_SPACES:   # the same vocabulary A8 built: English in words, your language in characters
    words = NMTDataset(A8_DIR / "train.tsv", tgt_vocab=CharVocab(), config=Config())
    words.tgt_vocab.build_vocab(words.tgt_texts)
else:
    words = NMTDataset(A8_DIR / "train.tsv", config=Config())   # the same vocabulary A8 built

# Your guess, before running: the percentage of target-side test tokens the word vocabulary
# cannot represent. Inflected languages run higher. Any number; the point is to have one.
EXPECTED_UNK_PERCENT = None     # <- e.g. 5.0


def unknown_rate(vocab, sentences):
    """Return (unknown tokens, all tokens) for sentences encoded with vocab.

    TODO: write it. vocab.encode(s, add_special_tokens=False) gives the token ids of one
    sentence, and vocab.unk_idx is the id every unknown word becomes. Over all the sentences,
    count how many ids are unk_idx and how many ids there are in all.
    """
    raise NotImplementedError("write unknown_rate, then run this cell again")


# A case whose answer you know: one made-up word is one unknown token out of one.
assert unknown_rate(words.src_vocab, ["qzxvw"]) == (1, 1), "unknown_rate is not counting right yet"
assert EXPECTED_UNK_PERCENT is not None, "write your guess in EXPECTED_UNK_PERCENT first"

word_unk = {}
for side, vocab in (("src", words.src_vocab), ("tgt", words.tgt_vocab)):
    unk, total = unknown_rate(vocab, test[side])
    word_unk[side] = unk
    units = UNITS if side == "tgt" else "word"
    print(f"{side}: {units} vocabulary {len(vocab):,};  test tokens it cannot represent: "
          f"{unk:,} of {total:,} ({100 * unk / total:.1f}%)")

print(f"\nYou guessed {EXPECTED_UNK_PERCENT:.1f}% on the target side.")
print(f"ln(V) for your target vocabulary: {uniform_loss(len(words.tgt_vocab)):.2f}")

The target-side percentage is the part to sit with. Those words are in your test references, so
every one of them is a word your model's translation is guaranteed to miss, and BLEU counts each
miss. No amount of training fixes it: the words are not in the vocabulary.

## Step 2: a subword vocabulary, trained on your training split only

SentencePiece learns a vocabulary of *pieces*: frequent words stay whole, and rare words are spelled
out of smaller pieces, down to single characters if need be. So any word made of characters it has
seen can be represented, including words it has never seen whole.

**Train it on `train.tsv` only.** Your test set must stay unseen, and a vocabulary is part of what a
model learns. Fitting it on everything would leak test material into the vocabulary, and nothing
would crash to tell you.

**The size is your choice.** 8,000 pieces is what the course model was measured with, and where
ln V is about 9.0, the number Lecture 7's slides quote. Fewer pieces (4,000): more words spelled
out, longer sequences, a smaller output layer. More pieces (16,000): more words kept whole,
shorter sequences, and more pieces seen too rarely to learn well. Pick one and say why in one
line; 8,000 is right if you have no reason to prefer another. Whichever you pick, A9 trains with
it, and the comparison is still one variable: your word vocabulary against this one. Training
takes about a minute.

In [ ]:
# Your choice. 8,000 is the course default; 4,000 and 16,000 are the alternatives worth a reason.
VOCAB_SIZE = 8_000
WHY = ""     # one line: why this size for your language

assert VOCAB_SIZE in (4_000, 8_000, 16_000), "pick one of the three sizes"
assert WHY.strip(), "say why, in one line, before training the vocabulary"

# A script with thousands of characters (Chinese, Japanese) cannot give every rare character a
# piece of its own at these sizes; 0.9995 lets the rarest fall back to <unk> instead of failing.
coverage = 0.9995 if NO_SPACES else None   # None: the library default, every character
train_sentencepiece([A8_DIR / "train.tsv"], model_prefix=str(A8_DIR / "spm"), vocab_size=VOCAB_SIZE,
                    character_coverage=coverage)
pieces = SentencePieceVocab(str(A8_DIR / "spm.model"))

print(f"Subword vocabulary: {len(pieces):,} pieces, shared by source and target.")
print(f"Why this size: {WHY.strip()}")
print(f"ln(V) = {uniform_loss(len(pieces)):.2f}")
print(f"Saved: {A8_DIR / 'spm.model'} and {A8_DIR / 'spm.vocab'}")

## Step 3: look at the pieces

`▁` marks the start of a word. A word that is one piece was common enough to earn its own entry;
a word split into several was rarer. Look for a word your word vocabulary called `<unk>`: here it
is spelled out of pieces instead of lost.

In [ ]:
label = UNITS.upper() + "S"
for sentence in test["tgt"].head(3):
    as_units = words.tgt_vocab.indices_to_tokens(words.tgt_vocab.encode(sentence, add_special_tokens=False))
    as_pieces = pieces.indices_to_tokens(pieces.encode(sentence, add_special_tokens=False))
    print(f"{label:10}", " ".join(as_units))
    print(f"{'PIECES':10}", " ".join(as_pieces))
    print()

## Step 4: the same count, with subwords

The same test sets, encoded with the pieces. The count should be zero. The only way it is not: a
*character* in the test set that never appears anywhere in training, such as a symbol or a letter
from another script. The cell names them if so.

In [ ]:
train = pd.read_csv(A8_DIR / "train.tsv", sep="\t", dtype=str, keep_default_na=False)

for side in ("src", "tgt"):
    unk, total = unknown_rate(pieces, test[side])
    units = UNITS if side == "tgt" else "word"
    print(f"{side}: test tokens the subword vocabulary cannot represent: {unk:,} of {total:,}   "
          f"({units} vocabulary: {word_unk[side]:,})")
    if unk:
        seen = set("".join(train["src"]) + "".join(train["tgt"]))
        unseen = sorted(set("".join(test[side])) - seen)
        print(f"   characters never seen in training: {unseen}")

## Step 5: nothing is lost on the way through

Pieces are only useful if they turn back into the same text. Ten test sentences, encoded and
decoded. One subtlety: SentencePiece normalizes text first (Unicode NFKC, and runs of spaces
collapse), so the check compares against the normalized sentence. A sentence that differs before
normalization differs only in how a character was typed, not in what it says.

In [ ]:
import unicodedata


def normalized(text):
    """What SentencePiece's default normalization does: NFKC, single spaces."""
    return " ".join(unicodedata.normalize("NFKC", text).split())


sample = list(test["tgt"].head(10))
round_trip = [pieces.decode(pieces.encode(s, add_special_tokens=False)) for s in sample]
exact = sum(a == b for a, b in zip(sample, round_trip))
after_normalizing = sum(normalized(a) == b for a, b in zip(sample, round_trip))

print(f"Round trip: {exact}/10 identical; {after_normalizing}/10 identical after normalizing.")
assert after_normalizing == 10, "a sentence changed beyond normalization; ask the teaching staff"
print(f"\nexample:\n  {sample[0]}\n  {' '.join(pieces.indices_to_tokens(pieces.encode(sample[0], add_special_tokens=False)))}\n  {round_trip[0]}")

## Step 6: the price, and the settings A9 needs

Pieces are shorter than words, so every sentence gets **longer**, measured in tokens. That costs
memory and time, and it moves one setting you have never had to think about: the **longest
translation the model is allowed to produce**. TorchLingo stops decoding at 100 tokens by default.
Under A8's word vocabulary that matched the 100-word cap. In pieces, 100 tokens is well short of
100 words, so a long translation would be cut off mid-sentence, and your A9 BLEU would drop for a
reason that has nothing to do with subwords. The cell measures your own longest sentence.
(**Without spaces**, it runs the other way: a piece is often several characters, so your
sentences get *shorter*. The setting is printed either way.)

Then, with the two `<unk>` counts and the ratio in front of you, **predict A9**: will BLEU go
up or down against your A8 score, by roughly how much, and why? The cell prints the prediction
with the settings, and the A9 write-up asks for it back, with one sentence reconciling it with
what happened. A prediction you wrote before training is worth more than any explanation
written after.

In [ ]:
if NO_SPACES:
    unit_len = train["tgt"].str.replace(" ", "").str.len()     # A8's tokens were characters
else:
    unit_len = train["tgt"].str.split().str.len()
piece_len = train["tgt"].map(lambda s: len(pieces.encode(s, add_special_tokens=False)))

MAX_DECODE = int(piece_len.max()) + 10    # your longest target, in pieces, with room to spare

print(f"Target sentences are {piece_len.sum() / unit_len.sum():.2f}x as many pieces as {UNITS}s.")
print(f"Longest training target: {unit_len.max()} {UNITS}s, {piece_len.max()} pieces.")

# Your prediction for A9, in one or two sentences: BLEU up or down against A8, by roughly how
# much, and why. Use the two <unk> counts and the ratio above.
BLEU_PREDICTION = ""

assert BLEU_PREDICTION.strip(), "write your prediction before the settings are printed"

print()
print("=== For Assignment 9: change these in your A8 notebook's Step 6 ===")
print(f"  In Config(...), add:              max_decode_length={MAX_DECODE},")
print(f"  In create_dataloaders(...), add:  use_sentencepiece=True,")
print(f"                                    sp_model_path=str(OUT_DIR / 'spm.model'),")
print(f"                                    sp_tgt_model_path=str(OUT_DIR / 'spm.model'),")
print("  (The same file twice: one subword model serves both languages.)")
if NO_SPACES:
    print("  Leave NO_SPACES = True: the pieces replace your characters, and your BLEU stays spBLEU,")
    print("  so the A8 and A9 scores are comparable.")
print()
print("=== And three names, so A9 does not overwrite your A8 model and translations ===")
print('  In Step 7:            experiment_name="assignment-9",  and  BEST_DIR = OUT_DIR / "best-a9"')
print('  In the scoring cell:  OUT_DIR / "best"  becomes  OUT_DIR / "best-a9",')
print('                        and  "test.hyp"   becomes  "test-a9.hyp"')
print("Everything else stays: the same files, the same split, the same training and scoring cells.")
print()
print(f"Your A9 prediction, for the write-up: {BLEU_PREDICTION.strip()}")

## What A9 is, now that you have seen this

Assignment 9 retrains your A8 model with the subword vocabulary and compares the two BLEU scores.
Because it uses **the same files and the same split**, the tokenizer is the only thing that
changes, so the difference in BLEU is the tokenizer's doing. That is the whole point of the
comparison, and why nothing here re-splits your data.

Three rules for the comparison to mean anything:

- **Score the translations `translate_batch` returns.** They are ordinary text: the pieces are
  decoded back into words before you see them. Never score token ids or pieces, which are not
  comparable with A8's word-level BLEU.
- **Set `max_decode_length` as printed above**, or long translations are cut short and the
  comparison is lost to truncation.
- **Keep A8's results.** A8 saved its best model in `best/` and its translations in `test.hyp`,
  under the training run named `assignment-8`. Run unchanged, A9 would reuse those names and
  write over them, and the write-up needs A8's translations next to A9's. The three new names
  printed above give A9 its own.

Your A8 notebook's scoring cell already does the first; A9 needs the three settings and the
three names printed above.

The A9 write-up asks for: your A8 and A9 BLEU, decoded the same way and saying how; your
prediction as printed above, and one sentence reconciling it with the result; the two
target-side `<unk>` rates and the piece-per-word ratio; your vocabulary size and why; and one
test sentence where the two systems' outputs differ, with your opinion of which is better.

---
## Report back

We will build a table on the board.

1. **Your target-side `<unk>` percentage, and your guess.** Who was furthest off, and which
   languages in the room run highest?
2. **Your vocabulary size, and why.** Anyone not at 8,000?
3. **Your piece-per-word ratio**, and what it does to `max_decode_length`.
4. **Your A9 prediction**, in one sentence. We will come back to these when A9 is in.